---
title: "Parallel Research and Fan-In"
draft: true
categories: [agents, workflows, langgraph]
---

Evidence collection is often a map-reduce workflow. Queries or sources can be processed independently, while synthesis must wait for all required branches and retain every result. LangGraph’s `Send` API expresses this fan-out explicitly.

## Map state from the question

The planner emits a bounded set of research tasks. Each task includes a query, source constraints, expected claim types, and a stable task ID. A `Send` route creates one branch per task. The branch returns evidence keyed by task ID rather than mutating shared objects.

## Fan-in semantics

Reducers combine branch outputs. The join node checks completeness, duplicate evidence, source diversity, and contradictory claims before synthesis. Missing branches are represented as failures or gaps, never silently dropped.

## Parallelism changes the failure surface

Parallel branches reduce wall-clock time but increase cost, rate-limit pressure, checkpoint volume, and reconciliation work. A bounded fan-out is part of the workflow contract. Per-thread stateful subgraphs also have concurrency constraints, so persistence configuration must be tested with the chosen topology.

## Deliverable and experiment

Implement map-reduce collection with `Send`, a provenance-aware reducer, and a join validator. Run the same fixture questions with one, two, and four branches. Measure evidence coverage, contradiction detection, wall-clock latency, token cost, duplicate source rate, and behavior when one branch fails.

The final graph should be able to continue with an explicit gap or route back to planning. “One branch returned something” is not a valid definition of complete research.
